In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.vol_project.bronze_btc_1m")

In [0]:
silver = (
    bronze
    .withColumn(
        "open_time_ms",
        F.when(F.col("open_time") >= 1e15, (F.col("open_time") / 1000).cast("long"))
         .otherwise(F.col("open_time"))
    )
    .withColumn("ts", F.timestamp_millis("open_time_ms"))
    .select("ts", "open", "high", "low", "close", "volume",
            "number_of_trades", "taker_buy_base_volume")
    .dropDuplicates(["ts"])
)

display(silver.orderBy("ts"))

In [0]:
print("rows:", silver.count())
display(silver.agg(F.min("ts"), F.max("ts")))

In [0]:
silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.vol_project.silver_btc_1m")

In [0]:
silver_1m = spark.table("workspace.vol_project.silver_btc_1m")

bars_5m = (
    silver_1m
    .withColumn("bucket", F.window("ts", "5 minutes").getField("start"))
    .groupBy("bucket")
    .agg(
        F.max_by("close", "ts").alias("close"),
        F.sum("volume").alias("volume"),
        F.count("*").alias("n_minutes")
    )
)

display(bars_5m.orderBy("bucket"))

In [0]:
from pyspark.sql.window import Window

w = Window.orderBy("bucket")

silver_5m = (
    bars_5m
    .withColumn("prev_close", F.lag("close").over(w))
    .withColumn("prev_bucket", F.lag("bucket").over(w))
    .withColumn("log_ret", F.log(F.col("close") / F.col("prev_close")))
    .withColumn(
        "gap_minutes",
        (F.unix_timestamp("bucket") - F.unix_timestamp("prev_bucket")) / 60
    )
    .withColumn("is_gap", F.col("gap_minutes") != 5)
    .drop("prev_close", "prev_bucket")
)

display(silver_5m.orderBy("bucket"))

In [0]:
print("rows:", silver_5m.count())
print("gap rows:", silver_5m.filter("is_gap").count())
display(silver_5m.select(
    F.mean("log_ret"), F.stddev("log_ret"),
    F.min("log_ret"), F.max("log_ret")
))

In [0]:
display(
    silver_5m.filter("is_gap")
    .select("bucket", "gap_minutes", "log_ret")
    .orderBy(F.desc("gap_minutes"))
)

In [0]:
silver_5m.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.vol_project.silver_btc_5m")